# 0. Libraries 

In [ ]:
# ==============================
# Core libraries
# ==============================
import numpy as np
import pandas as pd
import time
import warnings
import multiprocessing
from datetime import date, datetime, timedelta

# Use all but one CPU core for parallel processing
num_cores = max(multiprocessing.cpu_count() - 1, 1)
print("Using", num_cores, "cores for parallel processing.")

warnings.filterwarnings("ignore")


# ==============================
# Visualisation
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")  # optional, just to make plots look nicer


# ==============================
# Statistical utilities (EDA, tests)
# ==============================
import scipy.stats as stats
from scipy.stats import chi2, chi2_contingency, f_oneway

# Statsmodels (OLS, ANOVA, etc.)
import statsmodels.api as sm
from statsmodels.formula.api import ols


# ==============================
# Preprocessing & feature engineering
# ==============================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Optional: dimensionality reduction
from sklearn.decomposition import PCA


# ==============================
# Modelling algorithms (base models)
# ==============================

# Linear models / GLM-style
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

# Tree-based and ensemble models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    StackingRegressor,
    VotingRegressor
)

from sklearn.tree import DecisionTreeRegressor

# Distance-based & kernel-based models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Neural network regressor
from sklearn.neural_network import MLPRegressor


# ==============================
# Gradient boosting libraries (external)
# ==============================
# XGBoost
try:
    import xgboost as xgb
    xgb_available = True
    print("XGBoost available.")
except ImportError:
    xgb_available = False
    print("XGBoost NOT available (install xgboost if you want to use it).")

# LightGBM
try:
    import lightgbm as lgb
    lgb_available = True
    print("LightGBM available.")
except ImportError:
    lgb_available = False
    print("LightGBM NOT available (install lightgbm if you want to use it).")

# CatBoost (optional; often slower but nice to try)
try:
    import catboost as cb
    cb_available = True
    print("CatBoost available.")
except ImportError:
    cb_available = False
    print("CatBoost NOT available (install catboost if you want to use it).")


# ==============================
# Model evaluation & selection
# ==============================
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ==============================
# Model interpretation tools
# ==============================
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay
)

# If you later want SHAP, you can add:
# import shap


# 1. Loading the Data & Basic Inspection

In [ ]:
# Importing the training dataset
train = pd.read_csv("ML_WP_data/train.csv")

# Basic structural info (includes shape, dtypes, non-null counts)
train.info()

# Quick look at the first rows
display(train.head())

# Calculate total number of NaN values in the DataFrame
total_train_nans = train.isna().sum().sum()
print("Total NaN values in the training DataFrame:", total_train_nans)

# ------------------------------
# Helper: missingness summary (train only)
# ------------------------------

def missing_summary(df, sort_by="Percent_Missing", ascending=False):
    """
    Build a table with:
    - Data type
    - Number of missing values
    - Percentage of missing values
    - Basic descriptive stats (mean, std, min, 25%, 50%, 75%, max) for numeric cols
    """

    n_rows = len(df)

    # Core missing-value info
    miss = df.isna().sum()
    miss = miss[miss > 0]  # keep only variables with at least one missing

    summary = pd.DataFrame({
        "Data_Type": df[miss.index].dtypes.astype(str),
        "Missing_Values": miss,
        "Percent_Missing": (miss / n_rows * 100).round(2)
    })

    # Descriptive stats for numeric columns
    numeric_cols = df[miss.index].select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[
            ["mean", "std", "min", "25%", "50%", "75%", "max"]
        ]
        summary = summary.join(desc, how="left")

    # Sort and print total
    summary = summary.sort_values(sort_by, ascending=ascending)
    print(f"Total variables with missing values: {summary.shape[0]}")

    return summary

# Compute missingness summary for train
missing_train = missing_summary(train)
display(missing_train.head(70))  # show first 70 rows; adjust as needed


# 2. Target columns definition

In [ ]:
target_cols = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]
print("Target columns:", target_cols)
print("Missing values per target:")
print(train[target_cols].isna().sum(), "\n")


# 3. Exploratory Data Analysis (EDA)

## 3.1 Missingness & descriptive statistics

In [ ]:
# Summary for variables with missing values
missing_summary = (
    pd.DataFrame({
        "Data_Type": train.dtypes,
        "Missing_Values": train.isnull().sum(),
        "Percent_Missing": (train.isnull().sum() / len(train) * 100).round(2)
    })
    .query("Missing_Values > 0")
    .sort_values(by="Missing_Values", ascending=False)
)

# Descriptive stats
summary_stats = train.describe(include="all").transpose()

# Merge missingness with basic stats
merged_summary = missing_summary.merge(
    summary_stats[["mean", "std", "min", "25%", "50%", "75%", "max"]],
    left_index=True,
    right_index=True,
    how="left"
)

print(merged_summary.to_string())
print(f"\nTotal variables with missing values: {len(missing_summary)}")


## 3.2 Outlier overview (numeric variables, excluding targets)

In [ ]:
# Numeric columns (excluding targets)
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in target_cols]

def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    return outliers

outlier_summary = {}
for col in numeric_cols:
    n_outliers = len(detect_outliers(train, col))
    outlier_summary[col] = n_outliers

outlier_df = pd.DataFrame(list(outlier_summary.items()), columns=["Variable", "Outlier_Count"])
outlier_df["Outlier_%"] = (outlier_df["Outlier_Count"] / len(train)) * 100
outlier_df.sort_values(by="Outlier_%", ascending=False, inplace=True)

display(outlier_df.head(20))


## 3.3 Target distributions (histogram, boxplot, Q-Q plot)

In [ ]:
fig, axes = plt.subplots(len(target_cols), 3, figsize=(15, 12))

for i, target_col in enumerate(target_cols):
    # Histogram
    axes[i, 0].hist(train[target_col].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[i, 0].set_title(f"{target_col} – Distribution")
    axes[i, 0].set_xlabel("Temperature (°C)")
    axes[i, 0].set_ylabel("Frequency")

    # Boxplot
    axes[i, 1].boxplot(train[target_col].dropna(), vert=True)
    axes[i, 1].set_title(f"{target_col} – Boxplot")
    axes[i, 1].set_ylabel("Temperature (°C)")

    # Q–Q plot vs normal
    stats.probplot(train[target_col].dropna(), dist="norm", plot=axes[i, 2])
    axes[i, 2].set_title(f"{target_col} – Q–Q Plot vs Normal")

plt.tight_layout()
plt.show()


## 3.4 Skewness and kurtosis of targets

In [ ]:
for t in target_cols:
    clean_series = train[t].dropna()
    clean_series = clean_series[np.isfinite(clean_series)]

    skew = stats.skew(clean_series)
    kurt = stats.kurtosis(clean_series)

    print(f"{t}: Skewness = {skew:.3f}, Kurtosis = {kurt:.3f}")


## 3.5 Missingness by hour (pattern + chi-square test)

In [ ]:
# Row-wise missing count
train["missing_count"] = train.isnull().sum(axis=1)

# Total missing values per hour
missing_by_hour = (
    train.groupby("hour")["missing_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Top hours with most missing values:\n")
print(missing_by_hour.head(10))

# Plot
plt.figure(figsize=(10, 4))
missing_by_hour.sort_index().plot(kind="bar", edgecolor="black")
plt.title("Total Missing Values by Hour of the Day")
plt.xlabel("Hour (0–23)")
plt.ylabel("Number of Missing Values")
plt.tight_layout()
plt.show()

# Chi-square goodness-of-fit: are missing values equally distributed by hour?
missing_by_hour = train.groupby("hour")["missing_count"].sum()
expected = [missing_by_hour.sum() / len(missing_by_hour)] * len(missing_by_hour)

chi2_stat = ((missing_by_hour - expected) ** 2 / expected).sum()
p_value = 1 - chi2.cdf(chi2_stat, df=len(missing_by_hour) - 1)

print(f"Chi-square statistic: {chi2_stat:.2f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("→ Missingness differs significantly by hour (reject H₀).")
else:
    print("→ No significant hourly difference (fail to reject H₀).")


## 3.6 Missingness by season (if available) + chi-square + ANOVA

In [ ]:
if "season" in train.columns:
    # Use lower-case if that's how season is coded in the data
    season_order = ["winter", "spring", "summer", "autumn"]

    missing_by_season = (
        train.groupby("season")["missing_count"]
        .sum()
        .reindex(season_order)
    )

    print("\nMissing values by season:")
    print(missing_by_season)

    plt.figure(figsize=(6, 4))
    missing_by_season.plot(kind="bar", color="darkorange", edgecolor="black")
    plt.title("Missing Values by Season")
    plt.xlabel("Season")
    plt.ylabel("Total Missing Values")
    plt.tight_layout()
    plt.show()

    # Chi-square test
    missing_by_season_noorder = train.groupby("season")["missing_count"].sum()
    total_missing = missing_by_season_noorder.sum()
    n_seasons = len(missing_by_season_noorder)
    expected = [total_missing / n_seasons] * n_seasons

    chi2_stat = ((missing_by_season_noorder - expected) ** 2 / expected).sum()
    df_chi = n_seasons - 1
    p_value = 1 - chi2.cdf(chi2_stat, df=df_chi)

    print("=== Chi-square Test for Seasonal Missingness ===")
    print(f"Chi-square statistic: {chi2_stat:.2f}")
    print(f"Degrees of freedom: {df_chi}")
    print(f"p-value: {p_value:.4f}")

    if p_value < 0.05:
        print("→ Missingness differs significantly by season (reject H₀).")
    else:
        print("→ No significant seasonal difference in missingness (fail to reject H₀).")

    # ANOVA: does mean missing_count differ by season?
    model = ols("missing_count ~ C(season)", data=train).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    print("\n=== ANOVA Test for Seasonal Missingness ===")
    print(anova_table)

else:
    print("Column 'season' not found in 'train': skipping seasonal missingness analysis.")
